# Question Answering with LangChain, OpenAI, and MultiQuery Retriever


<br>

## Intro

In this lab you'll build a Retrieval-Augmented Generation (RAG) chatbot backed by Elasticsearch and OpenAI, using LangChain's `MultiQueryRetriever` to improve retrieval quality.

**Why use a `MultiQueryRetriever`?** A single user question, phrased one way, might miss documents that use different wording for the same idea. [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) asks an LLM to rephrase the original question into several alternative versions, runs a separate search for each version, and merges the results — giving you a broader, more relevant set of documents than a single search would.

Here's how the workflow will look like:

- **Setup (once):** split a sample dataset of documents into passages (chunks), turn each passage into an embedding, and store them in a vector database (in this case, we'll use Elasticsearch).
- **At question time:** the `MultiQueryRetriever` generates a few variations of the question, searches Elasticsearch with each one, and combines the retrieved passages. Those passages are then passed to an LLM, which uses them as context to write the final answer.


And here's what to expect from this lab:

1. **Complete a guided demo** — You'll be given the overall structure and process to follow: connecting to Elasticsearch and OpenAI, loading and chunking a sample set of workplace documents, embedding and indexing them into a vector store, and wiring up a `MultiQueryRetriever` chain that expands a question into several variations before retrieving and answering. Some steps are already implemented for you, and you'll need to complete the rest (e.g. filling in missing parameters and logic) to get it fully working.
2. **Replicate it yourself with variations** — Then, you'll create **at least two new iterations** of the question-answering step (new questions and/or settings) to explore how `MultiQueryRetriever` behaves.

By the end of this lab, you'll understand how multi-query retrieval works and be able to apply it to your own RAG pipelines.

<br>

## Install and import dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-elasticsearch<0.3" "jq==1.12.0"

     ---------------------------------------- 0.0/71.9 kB ? eta -:--:--
     ----- ---------------------------------- 10.2/71.9 kB ? eta -:--:--
     -------------------------------------- 71.9/71.9 kB 790.7 kB/s eta 0:00:00
   ---------------------------------------- 0.0/409.6 kB ? eta -:--:--
   --------------------------------------  399.4/409.6 kB 12.6 MB/s eta 0:00:01
   ---------------------------------------- 409.6/409.6 kB 8.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/952.8 kB ? eta -:--:--
   --------------------------------------- 952.8/952.8 kB 58.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/65.1 kB ? eta -:--:--
   ---------------------------------------- 65.1/65.1 kB ? eta 0:00:00
   ---------------------------------------- 0.0/87.2 kB ? eta -:--:--
   ---------------------------------------- 87.2/87.2 kB ? eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

In this lab, you'll use Elasticsearch as the vector database and search engine that stores your documents and retrieves the most relevant ones in response to user queries.

**What is Elasticsearch?**

Elasticsearch is a distributed search and analytics engine designed for fast retrieval over large collections of data. In AI applications, it can store both text and vector embeddings, making it well suited for Retrieval-Augmented Generation (RAG) workflows where an LLM retrieves relevant context before generating a response.


**Getting started with Elastic Cloud:**

1. If you don't already have one, sign up for a free trial: [https://www.elastic.co/](https://www.elastic.co/).
2. During sign-up (or afterwards), create a deployment/project. Either a classic deployment or a serverless project will work for this lab.
3. Once your deployment/project is ready, you'll need to find your **Cloud ID** and create an **API key**. 

<br>

> To find your **Cloud ID** and create an **API key**, 
> follow the instructions on the link below.
> 
>   👇👇👇
>
> https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
>
<br>

In [3]:
#
#
# To find your Cloud ID and create an API key, 
# follow the instructions on the link below:
#
# 👇👇👇
#
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
#
#
ELASTIC_URL = getpass("Elasticsearch URL: ")
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_url=ELASTIC_URL,
    es_api_key=ELASTIC_API_KEY,
    index_name="ironhack-multiquery-lab", #give it a meaningful name
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [4]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

In [5]:
try:
    info = vectorstore.client.info()
    print("✅ Connected to Elasticsearch!")
    print("Cluster:", info["cluster_name"])
    print("Version:", info["version"]["number"])

except Exception as e:
    print("❌ Connection failed:", e)

✅ Connected to Elasticsearch!
Cluster: c514134208fe40b6af343677afbea518
Version: 9.7.0


### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [6]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata.update({
    "name": record.get("name"),
    "summary": record.get("summary"),
    "url": record.get("url"),
    "category": record.get("category"),
    "updated_at": record.get("updated_at")
})

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [10]:
from langchain_openai import ChatOpenAI

documents = ElasticsearchStore.from_documents(
    docs,
    embedding=embeddings,
    index_name="ironhack-multiquery-lab",
    es_url=ELASTIC_URL,
    es_api_key=ELASTIC_API_KEY
)

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    openai_api_key=OPENAI_API_KEY
)

retriever = MultiQueryRetriever.from_llm(documents.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [11]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['Who comprises the NASA sales team and what are their roles?  ', 'Can you provide an overview of the NASA sales team and its functions?  ', "What information is available about the structure and responsibilities of NASA's sales team?"]


---- Answer ----
content='The NASA sales team refers to the sales organization covering the North America and South America regions within the broader Americas sales division of the company. Specifically, "NASA" stands for North America South America. This team is responsible for serving customers and pursuing business opportunities across countries in these two continents, including the United States, Canada, Mexico, as well as Central and South American countries.\n\nThe NASA sales team is led by two Area Vice-Presidents who oversee their respective subregions: Laura Martinez is the Area Vice-President for North America, and Gary Johnson is the Area Vice-President for South America. Together, they manage dedicated account managers, sales representatives, and support staff who work to identify new business opportunities, nurture existing client relationships, and ensure customer satisfaction throughout the Americas.\n\nIn summary, the NASA sales team is the combined sales force for No

In [12]:
LM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as short bullet point and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("Explain the Performance Management Policy.")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['What are the key components of the Performance Management Policy?  ', 'Can you provide an overview of how the Performance Management Policy works?  ', 'What guidelines and procedures are outlined in the Performance Management Policy?']


---- Answer ----
content='The Performance Management Policy is a formal framework designed to create a consistent, transparent, and fair process for evaluating, recognizing, and rewarding employee performance within the organization. Its primary purpose is to foster a culture of continuous improvement, professional growth, and open communication between employees and management. This policy applies to all employees and outlines several key components:\n\n1. **Performance Planning and Goal Setting:**  \n   At the start of each performance cycle, employees collaborate with their supervisors to set clear, achievable, and measurable goals. These goals are aligned with the company’s strategic objectives and consider the employee’s job responsibilities, professional development needs, and career aspirations. This collaborative goal-setting ensures that employees have a clear understanding of expectations and how their work contributes to the organization’s success.\n\n2. **Ongoing Feedback a

In [13]:
LM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as short bullet point and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("Explain how is the sales organization structured.")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['How is the sales team organized within the company?  ', 'What is the organizational hierarchy of the sales department?  ', 'Can you describe the structure of the sales division?']


---- Answer ----
content='The sales organization is structured regionally to effectively serve customers and achieve business objectives across multiple geographic areas. It is divided into four main regions, each led by an Area Vice-President who oversees the sales operations within their respective territories. Here is a detailed breakdown of the structure:\n\n1. **The Americas**  \n   This region includes the United States, Canada, Mexico, as well as Central and South America. It is further subdivided into North America and South America, each managed by an Area Vice-President:  \n   - **North America:** Led by Laura Martinez  \n   - **South America:** Led by Gary Johnson  \n   Together, they cover the entire Americas region, often referred to as NASA (North America South America).\n\n2. **Europe**  \n   The European sales team covers the entire continent, including major markets such as the United Kingdom, Germany, France, Spain, Italy, and others. This team focuses on understandin

In [14]:
LM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as short bullet point and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("Do we get tickets by the company to go to any NFL game?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['Does the company provide tickets for employees to attend NFL games?  ', 'Are NFL game tickets offered by the company for us to use?  ', 'Can employees receive NFL game tickets through the company?']


---- Answer ----
content="Based on the information provided in the retrieved context, there is no mention of the company providing tickets to any NFL games or any other sporting events. The documents focus primarily on updates to the company's work-from-home and hybrid work policies, specifying the number of days employees are required to work in the office and the need to coordinate schedules with supervisors and HR.\n\nTherefore, I do not have any information indicating that the company offers tickets to NFL games. If you are interested in such benefits, I would recommend reaching out directly to your HR department or checking any employee perks or benefits communications that might provide details on entertainment or event tickets." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 225, 'total_tokens': 349, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected

In [15]:
LM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as short bullet point and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("How is the weather in Madrid right now?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['What is the current weather condition in Madrid?  ', 'Can you provide the latest weather update for Madrid?  ', 'What is the present temperature and weather status in Madrid?']


---- Answer ----
content='The information provided in the retrieved context relates to company work-from-home policies and does not include any details about the current weather in Madrid. Therefore, I do not have the information needed to answer your question about the weather in Madrid right now. \n\nFor up-to-date weather information, I recommend checking a reliable weather website or app such as the Weather Channel, AccuWeather, or a local meteorological service. These sources provide real-time weather updates including temperature, precipitation, and forecasts.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 220, 'total_tokens': 317, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c2de8519c8', '

<br>

## 🚀 Your turn

Time to make this lab your own!


So far we've asked a single question ("what is the nasa sales team?") and looked at one answer. Now it's your turn to try the pipeline with your own questions and see how `MultiQueryRetriever` handles them.


Here's what to do:

1. **Pick 2 new questions** — Look through the sample dataset (or just get creative) and write down 2 questions you'd like the chatbot to answer, different from the example above.
2. **Ask each question** — For each question, copy the last code cell, replace the question passed to `chain.invoke(...)`, and run it.
3. **Check the logs** — Because logging is enabled, you'll see the alternative questions `MultiQueryRetriever` generated for you in the cell output. Take a look at them.
4. **Reflect** — In a markdown cell, briefly note: Did the generated questions look useful? Was the final answer correct and well grounded in the retrieved context, or did the model seem to be missing information?

<br>

💡 **Tip:**

- If an answer looks off, try asking the same question in a different way, or peek at the generated queries to see if they drifted from what you actually meant.

<br>

⭐️ **Bonus ideas:**

- Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?
- Look at the `k` parameter of `as_retriever()` (e.g. `vectorstore.as_retriever(search_kwargs={"k": 3})`) to control how many passages are retrieved per query, and see how that affects the final answer.
- Edit `LLM_CONTEXT_PROMPT` to change the tone or format of the final answer (e.g. ask for a short bullet-point summary instead of a verbose answer), and see how that changes the output.
- Ask a question that isn't covered by the sample dataset at all — does the chatbot correctly say it doesn't know, or does it make something up?

# Lab 2 — Chatbot with Multi-Query Retriever

## Objective

The objective of this lab was to build a Retrieval-Augmented Generation (RAG) chatbot using LangChain, OpenAI, and Elasticsearch.

The chatbot uses a MultiQueryRetriever to generate alternative versions of user questions, improving the retrieval of potentially relevant documents.

## Implementation

The system was developed using:

- **LangChain:** To build and connect the RAG components.
- **OpenAI Embeddings:** To convert documents into vector representations.
- **Elasticsearch:** To store and retrieve documents using vector similarity.
- **MultiQueryRetriever:** To generate multiple search queries from a single user question.
- **GPT-4.1-mini:** To generate answers based on retrieved context.
- **Prompt Templates:** To guide the model's responses.

## Testing and Results

Four questions were tested:

**Test 1 — Performance Management Policy**

The chatbot generated a detailed explanation of the company's performance management policy, covering performance evaluations, feedback, promotions, and employee development.

**Test 2 — Sales Organization Structure**

The chatbot explained the regional structure of the sales organization.

The MultiQueryRetriever successfully generated three alternative questions, demonstrating how query reformulation can support document retrieval.

**Test 3 — NFL Game Tickets**

The chatbot was asked whether employees receive tickets to NFL games.

Since the retrieved documents did not provide sufficient information, the chatbot correctly acknowledged that it could not confirm this benefit.

**Test 4 — Weather in Madrid**

The chatbot was asked about the current weather in Madrid.

Because the available documents were unrelated to weather information, the chatbot correctly stated that it could not answer the question.

## Key Learnings

- RAG combines document retrieval with language model generation.
- MultiQueryRetriever generates alternative queries to improve the coverage of document searches.
- Vector databases allow applications to retrieve documents based on semantic similarity.
- Prompt engineering helps guide response quality and reduce unsupported answers.
- A reliable chatbot should acknowledge when the available context is insufficient.
- Retrieved documents are not always relevant, making relevance filtering an important improvement.

## Challenges

The main technical challenge was updating the original implementation to work with the current Elasticsearch Cloud configuration and replacing the deprecated OpenAI model with GPT-4.1-mini.

Another challenge was controlling the length of generated responses, as the model sometimes produced detailed explanations despite instructions to remain concise.

## Future Improvements

- Implement document relevance filtering.
- Include source citations in generated answers.
- Improve prompt instructions for shorter responses.
- Evaluate retrieval quality and answer accuracy.
- Explore streaming responses to improve the user experience.

## Conclusion

This lab provided practical experience in building a functional RAG chatbot with LangChain and Elasticsearch.

All four test questions were processed successfully. The chatbot generated detailed answers when relevant information appeared to be available and acknowledged insufficient information when appropriate.

The results demonstrate the potential of MultiQueryRetriever for building more flexible question-answering applications, while also highlighting the importance of document relevance, source verification, and prompt engineering.